# DataMate agent: chạy bằng package `app`

Mọi logic nằm trong `app/` (xem README.md). Notebook này chỉ cấu hình và gọi.

**Chuẩn bị:** upload thư mục `datamate_agent/` thành một Kaggle Dataset và thêm vào notebook. Bản Gemini: Internet = On + Secret `GEMINI_API_KEY`, không cần GPU. Bản local: Accelerator = GPU T4 x2 + Internet = On. Trên Colab: đặt `ROOT` thành thư mục chứa `app/`.


In [ ]:
import glob, subprocess, sys
ROOT = glob.glob("/kaggle/input/**/datamate_agent/app/__init__.py", recursive=True)[0].rsplit("/app/", 1)[0]
sys.path.insert(0, ROOT)
BACKEND = "gemini"          # "gemini" | "local"
if BACKEND == "gemini":
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "google-genai"], check=False)
else:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "bitsandbytes", "accelerate", "peft"], check=False)


In [ ]:
from app import Config
from app.agent.pipeline import run_inference
from app.llm import create_backend
from evaluation.run_eval import run_train

if BACKEND == "gemini":
    cfg = Config(backend="gemini", max_biz=2, per_biz=15, limit_test=20)      # ~30 câu train + 20 câu test
else:
    cfg = Config(backend="local", max_biz=1, per_biz=10, limit_test=20)       # đo thời gian một câu trước đã
backend = create_backend(cfg)


## Chạy + chấm PA trên một phần train

In [ ]:
solver, split, ev, subs, logs = run_train(cfg, backend)

Các câu sai (mỗi câu −1) kèm code đã sinh:

In [ ]:
q_text = split.questions.set_index("id")["question"]
by_id = {lg["id"]: lg for lg in logs}
for r in ev[ev.status == "wrong"].head(15).itertuples():
    print(f"[{r.id}] {q_text[r.id]}\n   pred=({r.type}) {r.answer!r} | gold=({r.gold_type}) {r.gold!r} | agree={r.agree}")
    if by_id[r.id]["codes"]: print("   code:", by_id[r.id]["codes"][0][:600].replace("\n", "\n         "))


## (tùy chọn) Xuất dữ liệu SFT từ lời giải đã đúng trên train

In [ ]:
from app.agent.context import ContextBuilder
from training.export_sft import export_sft
n = export_sft(ev, logs, split, ContextBuilder(cfg, backend.count_tokens), f"{cfg.out_dir}/sft_train.jsonl")
print("Đã ghi", n, "mẫu SFT")

## Dự đoán public test và xuất `submission.jsonl` (đủ mọi id, có kiểm định)

In [ ]:
subs_te, logs_te, problems = run_inference(cfg, backend)